# HH Goa 2026 — Voice-RAG backend on Colab

Runs the FastAPI backend and exposes it on a public URL via a Cloudflare tunnel.

**Why Colab:** the backend needs ~2.3GB RAM. The jina reranker is ~1.3GB of that and is
*not* optional — its score is the working off-topic guardrail. Every 512MB free PaaS tier
OOMs. Colab gives ~13GB.

### Before you run
Add these in the **🔑 Secrets** panel (left sidebar), each with **Notebook access ON**:

`QDRANT_URL` · `QDRANT_API_KEY` · `GROQ_API_KEY` · `SARVAM_API_KEY` · `NIM_API_KEY`

Then **Runtime → Run all**. Takes ~6 minutes.

*Do not save this notebook back to GitHub* — it will overwrite the maintained version.


## 1 · Clone


In [ ]:
import os, subprocess, sys

REPO = "https://github.com/Rohit-0612/HHGoa26-Voice-Rag.git"
APP  = "/content/app"

if os.path.isdir(f"{APP}/.git"):
    subprocess.run(["git","-C",APP,"fetch","--all","-q"])
    subprocess.run(["git","-C",APP,"reset","--hard","origin/main","-q"])
    print("updated existing clone")
else:
    r = subprocess.run(["git","clone","-q",REPO,APP], capture_output=True, text=True)
    print("cloned" if r.returncode == 0 else f"CLONE FAILED:\n{r.stderr}")

os.chdir(APP)
sys.path.insert(0, APP)
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir(APP))[:8], "...")


## 2 · Install

Uses `requirements-serve.txt` (runtime only). It deliberately omits `datasets`,
`pyarrow` and `matplotlib` — installing those upgrades numpy/pyarrow and breaks the
running Colab session until a restart. Errors are printed in full here rather than
swallowed, so a failure is visible.


In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-serve.txt"],
    capture_output=True, text=True)

print("exit code:", r.returncode)
if r.returncode != 0:
    print("---- STDOUT ----"); print(r.stdout[-3000:])
    print("---- STDERR ----"); print(r.stderr[-3000:])
    raise SystemExit("pip install failed -- read the output above")

tail = (r.stdout or "").strip().splitlines()[-4:]
print("\n".join(tail) if tail else "installed cleanly")


### 2b · Verify the imports actually work


In [ ]:
import importlib, sys
sys.path.insert(0, "/content/app")

ok = True
for mod in ["fastapi","uvicorn","qdrant_client","fastembed","groq","openai",
            "pydantic_settings","httpx","numpy","multipart"]:
    try:
        importlib.import_module(mod); print(f"  OK      {mod}")
    except Exception as e:
        ok = False; print(f"  FAILED  {mod}: {type(e).__name__}: {e}")

print("\nall imports OK" if ok else "\nSOME IMPORTS FAILED -- see above")


## 3 · Secrets → .env

If anything reports MISSING, fix the name in the 🔑 panel (exact, uppercase) and make
sure *Notebook access* is toggled on for it, then re-run this cell.


In [ ]:
from google.colab import userdata
import pathlib

REQUIRED = ["QDRANT_URL","QDRANT_API_KEY","GROQ_API_KEY"]
OPTIONAL = ["SARVAM_API_KEY","NIM_API_KEY"]   # STT / LLM fallback

lines, missing_req, missing_opt = [], [], []
for k in REQUIRED + OPTIONAL:
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if v:
        lines.append(f"{k}={v}")
        print(f"  OK       {k}")
    else:
        (missing_req if k in REQUIRED else missing_opt).append(k)
        print(f"  MISSING  {k}")

lines += [
    "QDRANT_COLLECTION=msmarco_xi",
    "GROQ_MODEL=openai/gpt-oss-20b",
    "MAX_TOKENS=1024",
    "DENSE_MODEL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "SPARSE_MODEL=Qdrant/bm25",
    "RERANK_MODEL=jinaai/jina-reranker-v2-base-multilingual",
    "NIM_MODEL=nvidia/nvidia-nemotron-nano-9b-v2",
    "CORS_ORIGINS=*",
]
pathlib.Path("/content/app/.env").write_text("\n".join(lines) + "\n")

if missing_req:
    raise SystemExit(f"Cannot start without: {missing_req}")
if missing_opt:
    print(f"\nNOTE: {missing_opt} missing -- "
          f"{'voice input' if 'SARVAM_API_KEY' in missing_opt else ''}"
          f"{' and ' if len(missing_opt)>1 else ''}"
          f"{'LLM fallback' if 'NIM_API_KEY' in missing_opt else ''} will be disabled. "
          f"Text queries still work.")
print("\nwrote .env")


## 4 · Download models (~2 min)\n\nDone now so the first real request is fast.


In [ ]:
import time
t = time.time()
from fastembed import TextEmbedding, SparseTextEmbedding
from fastembed.rerank.cross_encoder import TextCrossEncoder

TextEmbedding('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print(f"  dense   ok  ({time.time()-t:.0f}s)")
SparseTextEmbedding('Qdrant/bm25')
print(f"  sparse  ok  ({time.time()-t:.0f}s)")
TextCrossEncoder('jinaai/jina-reranker-v2-base-multilingual')
print(f"  rerank  ok  ({time.time()-t:.0f}s)")
print("\nmodels cached")


## 5 · Start the server


In [ ]:
import subprocess, sys, time, os

os.chdir("/content/app")
LOG = "/content/server.log"

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.api.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, cwd="/content/app")

ready = False
for _ in range(150):                 # up to ~7.5 min: models load on startup
    time.sleep(3)
    log = open(LOG).read()
    if "Application startup complete" in log:
        ready = True; break
    if server.poll() is not None:    # process died
        print("SERVER DIED -- log:\n"); print(log[-4000:]); break

if ready:
    print("backend ready")
else:
    print("NOT READY. Last 3000 chars of log:\n")
    print(open(LOG).read()[-3000:])


## 6 · Public tunnel


In [ ]:
import subprocess, time, re, urllib.request, os, stat

BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(BIN):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
    print("cloudflared installed")

TLOG = "/content/tunnel.log"
tunnel = subprocess.Popen([BIN, "tunnel", "--url", "http://localhost:8000",
                           "--no-autoupdate"],
                          stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG).read())
    if m:
        url = m.group(0); break

print("\n" + "=" * 66)
if url:
    print("OPEN THIS LINK -- it serves the UI *and* the API:")
    print("   ", url)
    print("\nSame origin, so there is no CORS step and nothing to paste.")
else:
    print("TUNNEL FAILED. Log:\n")
    print(open(TLOG).read()[-2500:])
print("=" * 66)


## 7 · Verify end to end


In [ ]:
import requests, json, time

# A freshly-created trycloudflare hostname takes a few seconds to appear in
# public DNS. Checking immediately raises NameResolutionError even though the
# tunnel is fine, so wait for propagation instead of failing.
h = None
for attempt in range(20):
    try:
        r = requests.get(f"{url}/health", timeout=90)
        h = r.json()
        print(f"HTTP {r.status_code}  (ready after {attempt*5}s)")
        break
    except requests.exceptions.ConnectionError:
        print(f"\r  waiting for DNS to propagate... {attempt*5}s", end="", flush=True)
        time.sleep(5)

if h is None:
    print("\n\nStill unreachable. The tunnel may have dropped -- re-run cell 6.")
    print(open("/content/tunnel.log").read()[-1500:])
else:
    print(json.dumps(h, indent=1)[:500])
    print()
    print("  Qdrant reachable :", h.get("points", 0) > 0, f"({h.get('points'):,} points)")
    print("  LLM providers    :", h.get("providers"))
    print("  STT enabled      :", h.get("stt_enabled"))
    print("  Guardrail loaded :", h.get("scope_guard_enabled"))

    q = requests.post(f"{url}/query",
                      json={"text": "कॉर्पोरेशन क्या है?", "language": "hi"},
                      timeout=300).json()
    print("\n  answer   :", q.get("answer", "")[:90])
    print("  conf     :", q.get("confidence"), "| cites:", len(q.get("citations", [])))
    print("  total_ms :", round(q.get("timing", {}).get("total_ms", 0)))


## 8 · Keep alive

Run this and **leave the tab open**. It pings `/health` every 5 minutes so Colab does
not idle out. Colab still hard-stops at ~12h — re-run the notebook for a fresh URL.


In [ ]:
import time, requests
n = 0
while True:
    try:
        requests.get(f"{url}/health", timeout=60); n += 1
        print(f"\r alive {n} pings ({n*5} min)", end="", flush=True)
    except Exception as e:
        print(f"\n ping failed: {type(e).__name__}")
    time.sleep(300)
